# 00 - Setup and first checks

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pbtamarona/openpytea_workshop/blob/main/notebooks/00_setup_check.ipynb)

Run this notebook **before the workshop** to verify that your laptop is ready. Use it together with [INSTRUCTIONS.md](../INSTRUCTIONS.md). Every check prints `OK` when it passes. If a check fails, the message tells you which installation step to repeat.

Run it **at home before the workshop** (5 minutes). At the start of the session we only confirm that it passes.

## Learning goals
- Confirm that the notebook runs in the environment you created for the workshop.
- Confirm that `openpytea` and its scientific-Python dependencies import correctly.
- Run a five-line TEA and produce one plot, so that the whole toolchain is exercised once.
- Confirm that the `openpytea` command-line tool is on your PATH (used in notebook 06).

## 1) Environment snapshot

In [ ]:
import platform
import sys
from pathlib import Path

print("Python version   :", sys.version.split()[0])
print("Python executable:", sys.executable)
print("Platform         :", platform.platform())
print("Current folder   :", Path.cwd())

major, minor = sys.version_info[:2]
print("OK: Python 3.10 or newer" if (major, minor) >= (3, 10) else "FAIL: OpenPyTEA needs Python 3.10 or newer")

## 2) Import checks

All of these must print `OK`. If `openpytea` fails, go back to step 3 of [INSTRUCTIONS.md](../INSTRUCTIONS.md)
and make sure the notebook kernel uses the environment where you installed it (menu **Kernel**, then **Change Kernel**).
On Google Colab, run the next cell twice: the first run installs the package.

In [ ]:
# Run this cell first. It installs OpenPyTEA if it is missing (for example on Google Colab).
try:
    import openpytea
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "openpytea"])
    import openpytea

print("OpenPyTEA version:", openpytea.__version__)

In [ ]:
import importlib

for name in ["openpytea", "numpy", "pandas", "scipy", "matplotlib", "tqdm"]:
    try:
        module = importlib.import_module(name)
        print(f"OK   {name:<11} version {getattr(module, '__version__', 'unknown')}")
    except Exception as exc:
        print(f"FAIL {name:<11} {type(exc).__name__}: {exc}")

In [ ]:
from packaging.version import Version  # ships with pip/setuptools

import openpytea

print("OK: OpenPyTEA version is recent enough" if Version(openpytea.__version__) >= Version("3.1.0")
      else f"FAIL: found {openpytea.__version__}, please run  pip install --upgrade openpytea")

## 3) A five-line techno-economic assessment

This exercises the full chain: cost correlation lookup, plant economics, and plotting.
A small figure should appear below the cell.

In [ ]:
from openpytea import Equipment, Plant, fixed_capital_data, plot_stacked_bar

compressor = Equipment(name="C-101", param=1_000, process_type="Fluids",
                       category="Compressors, fans, & blowers", type="Compressor, centrifugal")
plant = Plant({
    "plant_name": "Setup check", "process_type": "Fluids", "equipment": [compressor],
    "plant_products": {"product": {"production": 10_000, "price": 3.0}},
    "variable_opex_inputs": {"electricity": {"consumption": 24_000, "price": 0.08}},
})
plant.calculate_all()
fig, ax = plot_stacked_bar(fixed_capital_data(plant))
print(f"OK: levelized cost = {plant.levelized_cost:.2f} USD/unit, NPV = {plant.npv:,.0f} USD")

## 4) Command-line tool (optional)

Installing `openpytea` also installs a command-line tool of the same name. It is used only in Exercise 6
(notebook 06), and a fallback is provided there, so a `WARNING` here is not a problem.

In [ ]:
import shutil

cli = shutil.which("openpytea")
print(f"OK: openpytea CLI found at {cli}" if cli else
      "WARNING: openpytea CLI not on PATH (only needed for notebook 06; `python -m openpytea.cli` works as a fallback)")

## Troubleshooting quick list

- `ModuleNotFoundError: openpytea`: the kernel is not the workshop environment. Change the kernel, or run
  `python -m ipykernel install --user --name openpytea-ws` with the environment activated and restart Jupyter Lab.
- No figure appears: run `%matplotlib inline` in a cell and re-run the plot cell.
- Anything else: email `P.B.Tamarona@tudelft.nl` with the full error message and your operating system.

## Recap

If every cell above printed `OK`, you are ready. See you at the workshop!